# NN-kNN Classification Workflow

This notebook uses the maintained IJCAI-26 NN-kNN core for classification. Retrieval, feature weighting, case scoring, and case normalization remain the current model; the output layer sums normalized case activation into class probability mass and trains it with negative log likelihood.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.classification_workflow import (
    list_supported_classification_datasets,
    list_supported_classification_benchmark_methods,
    make_classification_cfg,
    run_single_nnknn_classification_experiment,
    run_repeated_classification_model_benchmarks,
)

print(list_supported_classification_datasets())
print(list_supported_classification_benchmark_methods())

{'small': ['iris', 'zebra', 'zebra_special', 'wine', 'breast_cancer', 'balance', 'digits'], 'image': ['mnist', 'cifar10', 'svhn']}
{'small': ['nnknn', 'knn', 'mlp'], 'image': ['nnknn_conv_trainable', 'nnknn_conv_frozen', 'convnet', 'knn_pixels', 'knn_conv_frozen']}


## Small Dataset Sanity Check

The IJCAI-25 small-data family is available as `iris`, `zebra`, `zebra_special`, `wine`, `breast_cancer`, `balance`, and `digits`. Standardization is fitted on the training split only.

In [8]:
cfg = make_classification_cfg({
    "training_epochs": 400,
    "batch_size": 32,
    "patience": 10,
    "tau": 1.0,  # Representative small-data setting; retune per reporting protocol.
    "case_normalizer": "softmax",  # Change to "sparsemax" for sparse case activation.
    "top_k": 5,
    "explanation_mode": True,
    "checkpoint_path": "checkpoints/nnknn_classification_notebook.pth",
})

result = run_single_nnknn_classification_experiment(
    "zebra", cfg, run_seed=42, split_seed=42, checkpoint_label="iris_demo"
)
print("Validation accuracy:", result["accuracy"])
print("First class probability masses:", result["class_probabilities"][:3])

cases trainable: False
labels trainable: False
Number of feature extractor parameters: 0
Number of glocal weightor parameters: 1
torch.Size([1, 2])
Number of adapter parameters: 0
Number of case_net_params: 3
torch.Size([88])
torch.Size([88])
torch.Size([88, 1])
*****************
Training started for training_epochs epochs with batch size 32

[Stage 1] Retrieval-only training for up to 400 epochs (patience=10)
[Stage1] Epoch 1 - Val Acc: 0.5000 | Val Loss: 0.6792
[Stage1] New best (epoch 1) Val Loss: 0.6792 — saved: checkpoints\nnknn_classification_notebook_zebra_iris_demo_retr.pth
[Stage1] Epoch 2 - Val Acc: 0.5000 | Val Loss: 0.6792
[Stage1] New best (epoch 2) Val Loss: 0.6792 — saved: checkpoints\nnknn_classification_notebook_zebra_iris_demo_retr.pth
[Stage1] Epoch 3 - Val Acc: 0.5000 | Val Loss: 0.6792
[Stage1] New best (epoch 3) Val Loss: 0.6792 — saved: checkpoints\nnknn_classification_notebook_zebra_iris_demo_retr.pth
[Stage1] Epoch 4 - Val Acc: 0.5000 | Val Loss: 0.6792
[Stage1

In [9]:
# Explanation output uses cases retrieved by the current model.
print("Top retrieved class ids for query 0:", result["most_activated_class_ids"][0])
print("Top retrieved activation mass for query 0:", result["most_activated_activations"][0])

Top retrieved class ids for query 0: tensor([1, 1, 1, 1, 1])
Top retrieved activation mass for query 0: tensor([0.2007, 0.2005, 0.2003, 0.2002, 0.1982])


In [ ]:
import numpy as np
from ipywidgets import interact, IntSlider

import matplotlib.pyplot as plt

def plot_query_with_cases(query_idx):
    """Plot a single query and its top-k activated cases on 2D scatter plot."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Get all training data
    X_train = result["X_train"].numpy()
    y_train = result["y_train"].numpy()
    
    # Plot all training cases, color-coded by class
    colors = ['blue', 'red']
    for class_id in range(result["num_classes"]):
        mask = y_train == class_id
        ax.scatter(X_train[mask, 0], X_train[mask, 1], 
                  c=colors[class_id], label=f'Class {class_id}', 
                  alpha=0.5, s=50, edgecolors='black', linewidth=0.5)
    
    # Get query point
    query_point = result["X_val"][query_idx].numpy()
    ax.scatter(query_point[0], query_point[1], 
              c='green', marker='*', s=500, label='Query', 
              edgecolors='black', linewidth=1.5, zorder=5)
    
    # Get top-k activated cases and their activations
    top_cases = result["most_activated_cases"][query_idx].numpy()
    top_class_ids = result["most_activated_class_ids"][query_idx].numpy()
    top_activations = result["most_activated_activations"][query_idx].numpy()
    
    # Plot top-k cases with size proportional to activation
    for k, (case, class_id, activation) in enumerate(zip(top_cases, top_class_ids, top_activations)):
        size = 200 * activation  # Scale size by activation strength
        ax.scatter(case[0], case[1], c=colors[class_id], marker='s', 
                  s=size, alpha=0.7, edgecolors='gold', linewidth=2, 
                  label=f'Top-{k+1} (act={activation:.3f})', zorder=4)
    
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title(f'Query {query_idx} - Predicted Class: {result["predictions"][query_idx].item()}')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_query_with_cases, query_idx=IntSlider(min=0, max=10, step=1, value=0))

interactive(children=(IntSlider(value=0, description='query_idx', max=10), Output()), _dom_classes=('widget-in…

<function __main__.plot_query_with_cases(query_idx)>

In [11]:
glocal = result.get("glocal_weightor", result["model"].glocal_weightor)

if hasattr(glocal, "state_dict"):
    for name, value in glocal.state_dict().items():
        print(f"{name}: {value}")
elif hasattr(glocal, "weights"):
    print("weights:", glocal.weights)
else:
    print(glocal)

feature_weights: tensor([[ 1.7639, -0.0042]], device='cuda:0')


## Representative Benchmark Check

This short run checks that NN-kNN, kNN, and a four-hidden-layer MLP execute on shared stratified folds. Use more folds and epochs for a table intended for reporting.

In [ ]:
run_tabular_benchmark = False
if run_tabular_benchmark:
    summary, runs, _ = run_repeated_classification_model_benchmarks(
        "iris",
        cfg,
        methods=["nnknn", "knn", "mlp"],
        num_runs=3,
        mode="kfold",
        base_seed=42,
        method_cfgs={"mlp": {"epochs": 50, "patience": 10}},
    )
    display(summary)

## Image Workflow

For `mnist`, `cifar10`, or `svhn`, the loader uses the official train/test split and fits normalization statistics on training images. Training reserves an inner validation slice for checkpoint selection and reports on official test data. Start with a subset before a full image benchmark.

In [ ]:
run_image_demo = False
if run_image_demo:
    image_cfg = make_classification_cfg({
        "training_epochs": 5,
        "batch_size": 64,
        "top_k": 5,
        "explanation_mode": True,
        "checkpoint_path": "checkpoints/nnknn_mnist_subset.pth",
    })
    image_result = run_single_nnknn_classification_experiment(
        "mnist",
        image_cfg,
        dataset_kwargs={"max_train_samples": 1000, "max_eval_samples": 300},
        checkpoint_label="mnist_subset",
    )
    print("MNIST subset accuracy:", image_result["accuracy"])

For image baseline comparisons, call `run_repeated_classification_model_benchmarks` with methods `convnet`, `knn_pixels`, `knn_conv_frozen`, `nnknn_conv_trainable`, and `nnknn_conv_frozen`. The frozen NN-kNN method reuses a trained ConvNet feature extractor and keeps it frozen during NN-kNN training.